# Data Cleaning & Visualization Project

## Employee Dataset

**Objective:** Clean a raw employee dataset, handle missing values and duplicates, identify and treat salary outliers, and visualize useful patterns.

### Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Jupyter Notebook

### Workflow
1. Load and inspect the raw data
2. Check missing values and duplicates
3. Clean missing values
4. Remove duplicate records
5. Detect salary outliers using the IQR method
6. Cap extreme salary values for analysis while retaining the original salary
7. Create visualizations
8. Summarize key findings


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

df = pd.read_excel("sample_data_cleaning_project.xlsx")
df.head()


## 1. Initial Data Inspection

The dataset contains employee information such as name, age, salary, joining date, and department.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nSummary statistics:")
display(df.describe(include="all").T)


## 2. Missing Values and Duplicates

Missing numeric values are handled with the median because it is less sensitive to extreme values than the mean. Exact duplicate rows are removed.

In [ ]:
print("Missing values:")
display(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())


In [ ]:
clean_df = df.copy()

clean_df["Join_Date"] = pd.to_datetime(clean_df["Join_Date"], errors="coerce")

age_median = clean_df["Age"].median()
salary_median = clean_df["Salary"].median()

clean_df["Age"] = clean_df["Age"].fillna(age_median)
clean_df["Salary"] = clean_df["Salary"].fillna(salary_median)

before_duplicates = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
duplicates_removed = before_duplicates - len(clean_df)

print("Age median used:", age_median)
print("Salary median used:", salary_median)
print("Duplicate rows removed:", duplicates_removed)


## 3. Outlier Detection

The Interquartile Range (IQR) method is used on salary:

**Upper limit = Q3 + 1.5 × IQR**

The extreme values are retained in the original `Salary` column. A separate `Salary_Cleaned` column caps values above the upper limit so that visual summaries are not dominated by extreme observations.

In [ ]:
q1 = clean_df["Salary"].quantile(0.25)
q3 = clean_df["Salary"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

clean_df["Salary_Outlier"] = (
    (clean_df["Salary"] < lower_bound) |
    (clean_df["Salary"] > upper_bound)
)

clean_df["Salary_Cleaned"] = clean_df["Salary"].clip(
    lower=lower_bound, upper=upper_bound
)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Outlier count:", clean_df["Salary_Outlier"].sum())

display(clean_df.loc[clean_df["Salary_Outlier"], 
                     ["Name", "Salary", "Department", "Salary_Cleaned"]])


## 4. Feature Engineering

Joining year and month are extracted from `Join_Date` to support time-based analysis.

In [ ]:
clean_df["Join_Year"] = clean_df["Join_Date"].dt.year
clean_df["Join_Month"] = clean_df["Join_Date"].dt.month_name()

clean_df.head()


## 5. Visualizations

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["Age"].dropna(), bins=10)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
clean_df["Department"].value_counts().plot(kind="bar")
plt.title("Employees by Department")
plt.xlabel("Department")
plt.ylabel("Employee Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
clean_df.groupby("Department")["Salary_Cleaned"].mean().sort_values().plot(kind="bar")
plt.title("Average Cleaned Salary by Department")
plt.xlabel("Department")
plt.ylabel("Average Salary")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(clean_df["Salary"])
plt.title("Salary Distribution and Outliers")
plt.ylabel("Salary")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
clean_df.groupby("Join_Year").size().sort_index().plot(kind="bar")
plt.title("Employees Joining by Year")
plt.xlabel("Join Year")
plt.ylabel("Employee Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Key Findings

- The raw dataset contains missing values in Age and Salary.
- Exact duplicate records were identified and removed.
- Salary contains extreme high values that are flagged using the IQR method.
- Median imputation was used for missing Age and Salary values.
- Salary outliers are preserved in the original data and capped only in the analysis column.
- Department-level and joining-year visualizations make the employee distribution easier to interpret.

## Conclusion

The cleaning process improves consistency and makes the dataset suitable for analysis. Separating the original salary from the capped analytical salary preserves data transparency while reducing the influence of extreme values on summary visualizations.


In [ ]:
# Save the final cleaned dataset
clean_df.to_csv("cleaned_dataset_from_notebook.csv", index=False)
print("Saved cleaned_dataset_from_notebook.csv")
